<a href="https://colab.research.google.com/github/ubermanproductionsva-jpg/helix-rna-folding-engine/blob/main/Fork_of_HELIX_GEOMETRY_BASELINE_v1_ipynb_1e75ae.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# IMPORTANT: SOME KAGGLE DATA SOURCES ARE PRIVATE
# RUN THIS CELL IN ORDER TO IMPORT YOUR KAGGLE DATA SOURCES.
import kagglehub
kagglehub.login()


In [ ]:
# IMPORTANT: RUN THIS CELL IN ORDER TO IMPORT YOUR KAGGLE DATA SOURCES,
# THEN FEEL FREE TO DELETE THIS CELL.
# NOTE: THIS NOTEBOOK ENVIRONMENT DIFFERS FROM KAGGLE'S PYTHON
# ENVIRONMENT SO THERE MAY BE MISSING LIBRARIES USED BY YOUR
# NOTEBOOK.

stanford_rna_3d_folding_2_path = kagglehub.competition_download('stanford-rna-3d-folding-2')

print('Data source import complete.')


In [ ]:
# 1 — KAGGLE ENVIRONMENT + GLOBAL PATH REGISTRY (FINAL)

import os
import gc
import numpy as np
import pandas as pd

# --- FIND COMPETITION ROOT ---
BASE_CANDIDATES = [
    "/kaggle/input/competitions/stanford-rna-3d-folding-2",
    "/kaggle/input/stanford-rna-3d-folding-2",
]

COMP_ROOT = None
for p in BASE_CANDIDATES:
    if os.path.exists(p):
        COMP_ROOT = p
        break

if COMP_ROOT is None:
    raise FileNotFoundError(
        "❌ Competition dataset not found.\n"
        "👉 Click 'Add Input' and attach: Stanford RNA 3D Folding 2"
    )

# --- INPUT FILES ---
TRAIN_LABELS_PATH = os.path.join(COMP_ROOT, "train_labels.csv")
VALIDATION_LABELS_PATH = os.path.join(COMP_ROOT, "validation_labels.csv")
TRAIN_SEQS_PATH = os.path.join(COMP_ROOT, "train_sequences.csv")
VALIDATION_SEQS_PATH = os.path.join(COMP_ROOT, "validation_sequences.csv")
TEST_SEQS_PATH = os.path.join(COMP_ROOT, "test_sequences.csv")
SAMPLE_SUB_PATH = os.path.join(COMP_ROOT, "sample_submission.csv")

REQUIRED_FILES = [
    TRAIN_LABELS_PATH,
    VALIDATION_LABELS_PATH,
    TRAIN_SEQS_PATH,
    VALIDATION_SEQS_PATH,
    TEST_SEQS_PATH,
    SAMPLE_SUB_PATH,
]

missing_files = [p for p in REQUIRED_FILES if not os.path.exists(p)]
if missing_files:
    raise FileNotFoundError("❌ Missing required files:\n" + "\n".join(missing_files))

# --- WORKING OUTPUT PATHS ---
WORK_DIR = "/kaggle/working"
os.makedirs(WORK_DIR, exist_ok=True)

FEATURE_OUT = os.path.join(WORK_DIR, "FEATURE_TABLE__GEOMETRY_LABELS_V1.csv")
BENCH_OUT   = os.path.join(WORK_DIR, "HELIX_GEOMETRY_RECON_BENCHMARK_V1.csv")

# --- PRINT STATE ---
print("✅ Competition root:", COMP_ROOT)
print("✅ Working dir:", WORK_DIR)
print("✅ Feature output:", FEATURE_OUT)
print("✅ Benchmark output:", BENCH_OUT)

print("\n📁 Files in competition root:")
for f in os.listdir(COMP_ROOT):
    print(" -", f)

In [ ]:
# 2 — LOAD COMPETITION LABELS (MEMORY-SAFE, OPTIMIZED)

import pandas as pd
import numpy as np
import gc

USE_COLS = ["ID", "resname", "resid", "x_1", "y_1", "z_1", "chain", "copy"]

DTYPES = {
    "ID": "string",
    "resname": "category",
    "resid": "int32",
    "x_1": "float32",
    "y_1": "float32",
    "z_1": "float32",
    "chain": "category",
    "copy": "category",
}

print("📥 Loading train labels...")
df_labels = pd.read_csv(
    TRAIN_LABELS_PATH,
    usecols=USE_COLS,
    dtype=DTYPES,
    low_memory=False
)

print("📥 Appending validation labels...")
df_val = pd.read_csv(
    VALIDATION_LABELS_PATH,
    usecols=USE_COLS,
    dtype=DTYPES,
    low_memory=False
)

# Append WITHOUT creating large intermediate copies
df_labels = pd.concat([df_labels, df_val], ignore_index=True)
del df_val
gc.collect()

# --- DERIVE FIELDS ---
df_labels["target_id"] = df_labels["ID"].str.split("_").str[0]

df_labels["residue_index"] = df_labels["resid"].astype(np.int32)

df_labels["x"] = df_labels["x_1"]
df_labels["y"] = df_labels["y_1"]
df_labels["z"] = df_labels["z_1"]

# Drop original heavy columns EARLY
df_labels = df_labels.drop(columns=["ID", "resid", "x_1", "y_1", "z_1"])

gc.collect()

# --- SORT (critical for geometry) ---
df_labels = df_labels.sort_values(
    ["target_id", "chain", "copy", "residue_index"]
).reset_index(drop=True)

gc.collect()

# --- FINAL REPORT ---
print("✅ Label rows loaded:", len(df_labels))
print("✅ Targets:", df_labels["target_id"].nunique())
print("✅ Chains/copies:", df_labels.groupby(["target_id", "chain", "copy"]).ngroups)

print("\n📊 Memory usage (MB):", round(df_labels.memory_usage(deep=True).sum() / 1e6, 2))

df_labels.head(10)

In [ ]:
# 3 — ENFORCE SEQUENTIAL GEOMETRY + PHYSICAL STEP FILTER

group_cols = ["target_id", "chain", "copy"]

df_geo = df_labels.copy()

df_geo["resid_diff"] = df_geo.groupby(group_cols)["residue_index"].diff()
df_geo["is_seq"] = df_geo["resid_diff"].eq(1)

df_geo = df_geo[df_geo["resid_diff"].isna() | df_geo["is_seq"]].copy()

df_geo["dx"] = df_geo.groupby(group_cols)["x"].diff()
df_geo["dy"] = df_geo.groupby(group_cols)["y"].diff()
df_geo["dz"] = df_geo.groupby(group_cols)["z"].diff()
df_geo["step"] = np.sqrt(df_geo["dx"]**2 + df_geo["dy"]**2 + df_geo["dz"]**2)

df_geo = df_geo[
    df_geo["step"].isna() | ((df_geo["step"] >= 4.5) & (df_geo["step"] <= 8.0))
].copy()

steps = df_geo["step"].dropna().values
print("✅ Sequential geometry rows:", len(df_geo))
print("✅ Step mean:", float(steps.mean()))
print("✅ Step median:", float(np.median(steps)))
print("✅ Step p90:", float(np.percentile(steps, 90)))
print("✅ Step min:", float(steps.min()))
print("✅ Step max:", float(steps.max()))

display(df_geo.head(10))


In [ ]:
# 4 — BUILD GEOMETRY FEATURES (NORMALIZED DIRECTION + CURVATURE + TURN ANGLE)

df_feat = df_geo.copy()

df_feat["step_safe"] = df_feat["step"].replace(0, np.nan)
df_feat["dx_norm"] = df_feat["dx"] / df_feat["step_safe"]
df_feat["dy_norm"] = df_feat["dy"] / df_feat["step_safe"]
df_feat["dz_norm"] = df_feat["dz"] / df_feat["step_safe"]

df_feat["dx_prev"] = df_feat.groupby(group_cols)["dx"].shift(1)
df_feat["dy_prev"] = df_feat.groupby(group_cols)["dy"].shift(1)
df_feat["dz_prev"] = df_feat.groupby(group_cols)["dz"].shift(1)
df_feat["step_prev"] = df_feat.groupby(group_cols)["step"].shift(1)

df_feat["dx_prev_norm"] = df_feat["dx_prev"] / df_feat["step_prev"]
df_feat["dy_prev_norm"] = df_feat["dy_prev"] / df_feat["step_prev"]
df_feat["dz_prev_norm"] = df_feat["dz_prev"] / df_feat["step_prev"]

df_feat["curvature"] = (
    df_feat["dx_norm"] * df_feat["dx_prev_norm"] +
    df_feat["dy_norm"] * df_feat["dy_prev_norm"] +
    df_feat["dz_norm"] * df_feat["dz_prev_norm"]
)
df_feat["curvature"] = np.clip(df_feat["curvature"], -1.0, 1.0)
df_feat["turn_angle"] = np.arccos(df_feat["curvature"])

feature_cols = ["dx_norm", "dy_norm", "dz_norm", "curvature", "turn_angle", "step"]

df_feat = df_feat.dropna(subset=feature_cols).reset_index(drop=True)
df_feat.to_csv(FEATURE_OUT, index=False)

print("✅ Feature table rows:", len(df_feat))
print("✅ Feature columns:", feature_cols)
print("✅ Saved:", FEATURE_OUT)

display(df_feat[feature_cols].describe())


In [ ]:
# 5 — FEATURE BUILD (VECTORIZED, HIGH-PERFORMANCE, NO STALL)

import numpy as np
import pandas as pd
import gc

group_cols = ["target_id", "chain", "copy"]

chunks = []
total_groups = df_labels.groupby(group_cols).ngroups
processed = 0

print(f"🚀 Total groups: {total_groups}")

for (target_id, chain, copy), group in df_labels.groupby(group_cols):

    processed += 1
    if processed % 500 == 0:
        print(f"⏳ Processed {processed}/{total_groups}")

    group = group.sort_values("residue_index")

    coords = group[["x", "y", "z"]].values.astype(np.float32)

    if len(coords) < 3:
        continue

    # --- VECTORIZED DELTAS ---
    deltas = coords[1:] - coords[:-1]
    steps = np.linalg.norm(deltas, axis=1)

    norms = np.clip(steps, 1e-6, None)
    dirs = deltas / norms[:, None]

    # --- CURVATURE ---
    dot = np.sum(dirs[1:] * dirs[:-1], axis=1)
    dot = np.clip(dot, -1.0, 1.0)
    angles = np.arccos(dot)

    n = len(group) - 2
    if n <= 0:
        continue

    df_chunk = pd.DataFrame({
        "target_id": target_id,
        "chain": chain,
        "copy": copy,
        "residue_index": group["residue_index"].values[1:-1],

        "x": coords[1:-1, 0],
        "y": coords[1:-1, 1],
        "z": coords[1:-1, 2],

        "dx": deltas[:-1, 0],
        "dy": deltas[:-1, 1],
        "dz": deltas[:-1, 2],

        "step": steps[:-1],

        "dx_norm": dirs[:-1, 0],
        "dy_norm": dirs[:-1, 1],
        "dz_norm": dirs[:-1, 2],

        "turn_angle": angles,
        "curvature": angles
    })

    chunks.append(df_chunk)

# --- CONCAT ONCE ---
df_feat = pd.concat(chunks, ignore_index=True)

del chunks
gc.collect()

print("✅ Feature table built")
print("Rows:", len(df_feat))

In [ ]:
# 6 — SAVE FEATURE TABLE (CRITICAL CHECKPOINT)

import os

FEATURE_PATH = "/kaggle/working/FEATURE_TABLE__GEOMETRY_FULL.parquet"

df_feat.to_parquet(FEATURE_PATH, index=False)

print("✅ Feature table saved")
print("Path:", FEATURE_PATH)
print("Rows:", len(df_feat))

In [ ]:
# 7 — MEMORY RESET BEFORE TRAINING

import gc

del df_labels
gc.collect()

print("🧹 Memory cleared — ready for training")

In [ ]:
# 8 — BUILD TRAINING SET (DOUBLE-STEP FILTER — FINAL FIX)

import numpy as np

feature_cols = ["dx_norm", "dy_norm", "dz_norm", "curvature", "turn_angle", "step"]

# --- TARGETS ---
df_feat["dx_target"] = df_feat.groupby(["target_id", "chain", "copy"])["dx"].shift(-1)
df_feat["dy_target"] = df_feat.groupby(["target_id", "chain", "copy"])["dy"].shift(-1)
df_feat["dz_target"] = df_feat.groupby(["target_id", "chain", "copy"])["dz"].shift(-1)

# --- COMPUTE NEXT STEP ---
df_feat["step_target"] = np.sqrt(
    df_feat["dx_target"]**2 +
    df_feat["dy_target"]**2 +
    df_feat["dz_target"]**2
)

# --- DROP NA ---
df_model = df_feat.dropna(
    subset=feature_cols + ["dx_target", "dy_target", "dz_target", "step_target"]
).copy()

print("Before filtering:", len(df_model))

# 🔥 FILTER CURRENT STEP
df_model = df_model[
    (df_model["step"] > 4.0) &
    (df_model["step"] < 8.0)
]

print("After current step filter:", len(df_model))

# 🔥 FILTER TARGET STEP (THIS IS THE FIX)
df_model = df_model[
    (df_model["step_target"] > 4.0) &
    (df_model["step_target"] < 8.0)
]

print("After target step filter:", len(df_model))

# 🔥 ANGLE SAFETY
df_model = df_model[
    (df_model["turn_angle"] >= 0.0) &
    (df_model["turn_angle"] <= np.pi)
]

print("After angle filter:", len(df_model))

# 🔥 SAMPLE
SAMPLE_SIZE = 1_500_000

if len(df_model) > SAMPLE_SIZE:
    df_model = df_model.sample(SAMPLE_SIZE, random_state=42)

print("Final training rows:", len(df_model))

# --- FINAL SANITY ---
print("\n📊 STEP CHECK:")
print(df_model["step"].describe())

print("\n📊 TARGET STEP CHECK:")
print(df_model["step_target"].describe())

print("\n📊 TARGET DELTA CHECK:")
print(df_model[["dx_target","dy_target","dz_target"]].describe())

In [ ]:
# 9 — TRAIN RF DELTA MODELS (MEMORY-SAFE, SEQUENTIAL)

import numpy as np
import gc
from sklearn.ensemble import RandomForestRegressor

feature_cols = ["dx_norm", "dy_norm", "dz_norm", "curvature", "turn_angle", "step"]

X = df_model[feature_cols].values.astype(np.float32)
y_dx = df_model["dx_target"].values.astype(np.float32)
y_dy = df_model["dy_target"].values.astype(np.float32)
y_dz = df_model["dz_target"].values.astype(np.float32)

print("X shape:", X.shape)
print("Starting dx model...")

model_dx = RandomForestRegressor(
    n_estimators=80,
    random_state=42,
    n_jobs=-1,
    max_depth=20,
    min_samples_leaf=2,
)
model_dx.fit(X, y_dx)
print("✅ dx model trained")

gc.collect()

print("Starting dy model...")
model_dy = RandomForestRegressor(
    n_estimators=80,
    random_state=42,
    n_jobs=-1,
    max_depth=20,
    min_samples_leaf=2,
)
model_dy.fit(X, y_dy)
print("✅ dy model trained")

gc.collect()

print("Starting dz model...")
model_dz = RandomForestRegressor(
    n_estimators=80,
    random_state=42,
    n_jobs=-1,
    max_depth=20,
    min_samples_leaf=2,
)
model_dz.fit(X, y_dz)
print("✅ dz model trained")

gc.collect()

print("🎉 All RF models trained successfully")

In [ ]:
# 10 — BUILD EVALUATION SUBSET (CHAIN-LEVEL HOLDOUT, MEMORY-SAFE)

import pandas as pd
import numpy as np
import gc

group_cols = ["target_id", "chain", "copy"]

# Use the FULL feature table for evaluation candidate chains,
# but only evaluate on a limited number of chains to stay stable.
eval_keys = (
    df_feat[group_cols]
    .drop_duplicates()
    .sample(n=min(250, df_feat[group_cols].drop_duplicates().shape[0]), random_state=42)
    .reset_index(drop=True)
)

df_eval = df_feat.merge(eval_keys, on=group_cols, how="inner")

feature_cols = ["dx_norm", "dy_norm", "dz_norm", "curvature", "turn_angle", "step"]

df_eval["dx_target"] = df_eval.groupby(group_cols)["dx"].shift(-1)
df_eval["dy_target"] = df_eval.groupby(group_cols)["dy"].shift(-1)
df_eval["dz_target"] = df_eval.groupby(group_cols)["dz"].shift(-1)

df_eval = df_eval.dropna(subset=feature_cols + ["dx_target", "dy_target", "dz_target"]).copy()
df_eval = df_eval.sort_values(group_cols + ["residue_index"]).reset_index(drop=True)

print("✅ Evaluation chains:", eval_keys.shape[0])
print("✅ Evaluation rows:", len(df_eval))

gc.collect()

In [ ]:
# 11 — STREAMING RECONSTRUCTION BENCHMARK (DIRECTION-ONLY, TRUE-STEP ANCHORED)

import numpy as np
import pandas as pd
import gc

group_cols = ["target_id", "chain", "copy"]
feature_cols = ["dx_norm", "dy_norm", "dz_norm", "curvature", "turn_angle", "step"]

recon_rows = []
total_groups = df_eval.groupby(group_cols).ngroups
processed = 0

print(f"🚀 Evaluating {total_groups} chains")

for (target_id, chain, copy), group in df_eval.groupby(group_cols):
    processed += 1

    if processed % 25 == 0:
        print(f"⏳ Evaluated {processed}/{total_groups}")

    group = group.sort_values("residue_index").reset_index(drop=True)

    if len(group) < 4:
        continue

    coords_true = group[["x", "y", "z"]].values.astype(np.float32)
    Xg = group[feature_cols].values.astype(np.float32)

    pred_dx_g = model_dx.predict(Xg).astype(np.float32)
    pred_dy_g = model_dy.predict(Xg).astype(np.float32)
    pred_dz_g = model_dz.predict(Xg).astype(np.float32)

    true_steps = group["step"].values.astype(np.float32)

    coords_pred = [coords_true[0].copy()]

    for i in range(1, len(group)):
        direction = np.array(
            [pred_dx_g[i - 1], pred_dy_g[i - 1], pred_dz_g[i - 1]],
            dtype=np.float32
        )

        norm = np.linalg.norm(direction)

        if norm < 1e-6 or not np.isfinite(norm):
            # fallback to previous true local direction if prediction is degenerate
            direction = np.array(
                [group.iloc[i - 1]["dx"], group.iloc[i - 1]["dy"], group.iloc[i - 1]["dz"]],
                dtype=np.float32
            )
            norm = np.linalg.norm(direction)

            if norm < 1e-6 or not np.isfinite(norm):
                direction = np.array([1.0, 0.0, 0.0], dtype=np.float32)
                norm = 1.0

        direction = direction / norm

        true_step = true_steps[i - 1]
        step_vec = direction * true_step

        coords_pred.append(coords_pred[-1] + step_vec)

    coords_pred = np.asarray(coords_pred, dtype=np.float32)

    point_error = np.linalg.norm(coords_pred - coords_true, axis=1)

    recon_rows.append({
        "target_id": target_id,
        "chain": chain,
        "copy": copy,
        "residue_count": int(len(group)),
        "mean_error": float(point_error.mean()),
        "median_error": float(np.median(point_error)),
        "p90_error": float(np.percentile(point_error, 90)),
    })

    del pred_dx_g, pred_dy_g, pred_dz_g, coords_pred, point_error, Xg, coords_true, true_steps
    gc.collect()

df_recon = pd.DataFrame(recon_rows)

print("\n🧬 RECONSTRUCTION RESULTS")
print("Chains evaluated:", len(df_recon))
print("Mean error:", float(df_recon["mean_error"].mean()))
print("Median error:", float(df_recon["median_error"].median()))
print("P90:", float(np.percentile(df_recon["mean_error"], 90)))

display(df_recon.head(10))

In [ ]:
# 12 — WINDOWED RECONSTRUCTION BENCHMARK (DRIFT-CONTROLLED)

import numpy as np
import pandas as pd
import gc

group_cols = ["target_id", "chain", "copy"]
feature_cols = ["dx_norm", "dy_norm", "dz_norm", "curvature", "turn_angle", "step"]

WINDOW_SIZE = 10

recon_rows = []
total_groups = df_eval.groupby(group_cols).ngroups
processed = 0

print(f"🚀 Evaluating {total_groups} chains with window size = {WINDOW_SIZE}")

for (target_id, chain, copy), group in df_eval.groupby(group_cols):
    processed += 1

    if processed % 25 == 0:
        print(f"⏳ Evaluated {processed}/{total_groups}")

    group = group.sort_values("residue_index").reset_index(drop=True)

    if len(group) < 4:
        continue

    coords_true = group[["x", "y", "z"]].values.astype(np.float32)
    Xg = group[feature_cols].values.astype(np.float32)
    true_steps = group["step"].values.astype(np.float32)

    pred_dx_g = model_dx.predict(Xg).astype(np.float32)
    pred_dy_g = model_dy.predict(Xg).astype(np.float32)
    pred_dz_g = model_dz.predict(Xg).astype(np.float32)

    coords_pred = np.zeros_like(coords_true, dtype=np.float32)
    coords_pred[0] = coords_true[0].copy()

    # Windowed reconstruction:
    # Reset anchor to true coordinate at each window start to measure local directional quality
    for start in range(0, len(group), WINDOW_SIZE):
        end = min(start + WINDOW_SIZE, len(group))

        coords_pred[start] = coords_true[start].copy()

        for i in range(start + 1, end):
            direction = np.array(
                [pred_dx_g[i - 1], pred_dy_g[i - 1], pred_dz_g[i - 1]],
                dtype=np.float32
            )

            norm = np.linalg.norm(direction)

            if norm < 1e-6 or not np.isfinite(norm):
                fallback = np.array(
                    [group.iloc[i - 1]["dx"], group.iloc[i - 1]["dy"], group.iloc[i - 1]["dz"]],
                    dtype=np.float32
                )
                fallback_norm = np.linalg.norm(fallback)

                if fallback_norm < 1e-6 or not np.isfinite(fallback_norm):
                    direction = np.array([1.0, 0.0, 0.0], dtype=np.float32)
                    norm = 1.0
                else:
                    direction = fallback
                    norm = fallback_norm

            direction = direction / norm
            step_vec = direction * true_steps[i - 1]

            coords_pred[i] = coords_pred[i - 1] + step_vec

    point_error = np.linalg.norm(coords_pred - coords_true, axis=1)

    recon_rows.append({
        "target_id": target_id,
        "chain": chain,
        "copy": copy,
        "residue_count": int(len(group)),
        "window_size": int(WINDOW_SIZE),
        "mean_error": float(point_error.mean()),
        "median_error": float(np.median(point_error)),
        "p90_error": float(np.percentile(point_error, 90)),
    })

    del pred_dx_g, pred_dy_g, pred_dz_g, coords_pred, point_error, Xg, coords_true, true_steps
    gc.collect()

df_recon_windowed = pd.DataFrame(recon_rows)

print("\n🧬 WINDOWED RECONSTRUCTION RESULTS")
print("Chains evaluated:", len(df_recon_windowed))
print("Mean error:", float(df_recon_windowed["mean_error"].mean()))
print("Median error:", float(df_recon_windowed["median_error"].median()))
print("P90:", float(np.percentile(df_recon_windowed["mean_error"], 90)))

display(df_recon_windowed.head(10))

In [ ]:
# 13 — WINDOW SIZE SWEEP (LOCAL DRIFT DIAGNOSTIC)

import numpy as np
import pandas as pd
import gc

group_cols = ["target_id", "chain", "copy"]
feature_cols = ["dx_norm", "dy_norm", "dz_norm", "curvature", "turn_angle", "step"]

WINDOW_SIZES = [4, 6, 8, 10, 12, 16, 20]

all_rows = []

total_groups = df_eval.groupby(group_cols).ngroups
print(f"🚀 Evaluating {total_groups} chains across window sizes: {WINDOW_SIZES}")

for window_size in WINDOW_SIZES:
    print(f"\n🔍 Window size = {window_size}")

    recon_rows = []
    processed = 0

    for (target_id, chain, copy), group in df_eval.groupby(group_cols):
        processed += 1

        if processed % 25 == 0:
            print(f"⏳ Window {window_size}: evaluated {processed}/{total_groups}")

        group = group.sort_values("residue_index").reset_index(drop=True)

        if len(group) < 4:
            continue

        coords_true = group[["x", "y", "z"]].values.astype(np.float32)
        Xg = group[feature_cols].values.astype(np.float32)
        true_steps = group["step"].values.astype(np.float32)

        pred_dx_g = model_dx.predict(Xg).astype(np.float32)
        pred_dy_g = model_dy.predict(Xg).astype(np.float32)
        pred_dz_g = model_dz.predict(Xg).astype(np.float32)

        coords_pred = np.zeros_like(coords_true, dtype=np.float32)
        coords_pred[0] = coords_true[0].copy()

        for start in range(0, len(group), window_size):
            end = min(start + window_size, len(group))
            coords_pred[start] = coords_true[start].copy()

            for i in range(start + 1, end):
                direction = np.array(
                    [pred_dx_g[i - 1], pred_dy_g[i - 1], pred_dz_g[i - 1]],
                    dtype=np.float32
                )

                norm = np.linalg.norm(direction)

                if norm < 1e-6 or not np.isfinite(norm):
                    fallback = np.array(
                        [group.iloc[i - 1]["dx"], group.iloc[i - 1]["dy"], group.iloc[i - 1]["dz"]],
                        dtype=np.float32
                    )
                    fallback_norm = np.linalg.norm(fallback)

                    if fallback_norm < 1e-6 or not np.isfinite(fallback_norm):
                        direction = np.array([1.0, 0.0, 0.0], dtype=np.float32)
                        norm = 1.0
                    else:
                        direction = fallback
                        norm = fallback_norm

                direction = direction / norm
                step_vec = direction * true_steps[i - 1]
                coords_pred[i] = coords_pred[i - 1] + step_vec

        point_error = np.linalg.norm(coords_pred - coords_true, axis=1)

        recon_rows.append({
            "target_id": target_id,
            "chain": chain,
            "copy": copy,
            "residue_count": int(len(group)),
            "window_size": int(window_size),
            "mean_error": float(point_error.mean()),
            "median_error": float(np.median(point_error)),
            "p90_error": float(np.percentile(point_error, 90)),
        })

        del pred_dx_g, pred_dy_g, pred_dz_g, coords_pred, point_error, Xg, coords_true, true_steps
        gc.collect()

    df_window = pd.DataFrame(recon_rows)

    print("Chains evaluated:", len(df_window))
    print("Mean error:", float(df_window["mean_error"].mean()))
    print("Median error:", float(df_window["median_error"].median()))
    print("P90:", float(np.percentile(df_window["mean_error"], 90)))

    all_rows.append(df_window)

df_window_sweep = pd.concat(all_rows, ignore_index=True)

summary = (
    df_window_sweep
    .groupby("window_size")
    .agg(
        chains_evaluated=("mean_error", "count"),
        mean_error=("mean_error", "mean"),
        median_error=("median_error", "median"),
        p90_of_mean_error=("mean_error", lambda x: np.percentile(x, 90)),
    )
    .reset_index()
    .sort_values("mean_error")
)

print("\n🧬 WINDOW SWEEP SUMMARY")
display(summary)

WINDOW_SWEEP_OUT = "/kaggle/working/HELIX_WINDOW_SWEEP_V1.csv"
df_window_sweep.to_csv(WINDOW_SWEEP_OUT, index=False)

SUMMARY_OUT = "/kaggle/working/HELIX_WINDOW_SWEEP_SUMMARY_V1.csv"
summary.to_csv(SUMMARY_OUT, index=False)

print("\n💾 Saved:")
print(WINDOW_SWEEP_OUT)
print(SUMMARY_OUT)

In [ ]:
# 14 — SLIDING WINDOW RECONSTRUCTION (OVERLAP + AVERAGE STITCHING)

import numpy as np
import pandas as pd
import gc

group_cols = ["target_id", "chain", "copy"]
feature_cols = ["dx_norm", "dy_norm", "dz_norm", "curvature", "turn_angle", "step"]

WINDOW_SIZE = 4
STRIDE = 1

recon_rows = []
total_groups = df_eval.groupby(group_cols).ngroups
processed = 0

print(f"🚀 Evaluating {total_groups} chains with sliding windows")
print(f"Window size = {WINDOW_SIZE}, stride = {STRIDE}")

for (target_id, chain, copy), group in df_eval.groupby(group_cols):
    processed += 1

    if processed % 25 == 0:
        print(f"⏳ Evaluated {processed}/{total_groups}")

    group = group.sort_values("residue_index").reset_index(drop=True)

    if len(group) < WINDOW_SIZE:
        continue

    coords_true = group[["x", "y", "z"]].values.astype(np.float32)
    Xg = group[feature_cols].values.astype(np.float32)
    true_steps = group["step"].values.astype(np.float32)

    pred_dx_g = model_dx.predict(Xg).astype(np.float32)
    pred_dy_g = model_dy.predict(Xg).astype(np.float32)
    pred_dz_g = model_dz.predict(Xg).astype(np.float32)

    n = len(group)

    # Accumulate multiple overlapping coordinate votes per residue
    coord_sum = np.zeros((n, 3), dtype=np.float64)
    coord_count = np.zeros(n, dtype=np.float64)

    window_starts = list(range(0, n - WINDOW_SIZE + 1, STRIDE))
    if window_starts[-1] != n - WINDOW_SIZE:
        window_starts.append(n - WINDOW_SIZE)

    for start in window_starts:
        end = start + WINDOW_SIZE

        # Anchor each window at its true start coordinate
        window_coords = [coords_true[start].astype(np.float32).copy()]

        for i in range(start + 1, end):
            direction = np.array(
                [pred_dx_g[i - 1], pred_dy_g[i - 1], pred_dz_g[i - 1]],
                dtype=np.float32
            )

            norm = np.linalg.norm(direction)

            if norm < 1e-6 or not np.isfinite(norm):
                fallback = np.array(
                    [group.iloc[i - 1]["dx"], group.iloc[i - 1]["dy"], group.iloc[i - 1]["dz"]],
                    dtype=np.float32
                )
                fallback_norm = np.linalg.norm(fallback)

                if fallback_norm < 1e-6 or not np.isfinite(fallback_norm):
                    direction = np.array([1.0, 0.0, 0.0], dtype=np.float32)
                    norm = 1.0
                else:
                    direction = fallback
                    norm = fallback_norm

            direction = direction / norm
            step_vec = direction * true_steps[i - 1]

            window_coords.append(window_coords[-1] + step_vec)

        window_coords = np.asarray(window_coords, dtype=np.float32)

        # Stitch by averaging overlapping coordinate votes
        coord_sum[start:end] += window_coords.astype(np.float64)
        coord_count[start:end] += 1.0

    # Safety fallback for uncovered residues
    uncovered = coord_count == 0
    if np.any(uncovered):
        coord_sum[uncovered] = coords_true[uncovered]
        coord_count[uncovered] = 1.0

    coords_pred = (coord_sum / coord_count[:, None]).astype(np.float32)

    point_error = np.linalg.norm(coords_pred - coords_true, axis=1)

    recon_rows.append({
        "target_id": target_id,
        "chain": chain,
        "copy": copy,
        "residue_count": int(len(group)),
        "window_size": int(WINDOW_SIZE),
        "stride": int(STRIDE),
        "mean_error": float(point_error.mean()),
        "median_error": float(np.median(point_error)),
        "p90_error": float(np.percentile(point_error, 90)),
    })

    del pred_dx_g, pred_dy_g, pred_dz_g, coords_pred, point_error, Xg, coords_true, true_steps, coord_sum, coord_count
    gc.collect()

df_recon_sliding = pd.DataFrame(recon_rows)

print("\n🧬 SLIDING WINDOW RECONSTRUCTION RESULTS")
print("Chains evaluated:", len(df_recon_sliding))
print("Mean error:", float(df_recon_sliding["mean_error"].mean()))
print("Median error:", float(df_recon_sliding["median_error"].median()))
print("P90:", float(np.percentile(df_recon_sliding["mean_error"], 90)))

display(df_recon_sliding.head(10))

SLIDING_OUT = "/kaggle/working/HELIX_SLIDING_WINDOW_RECON_V1.csv"
df_recon_sliding.to_csv(SLIDING_OUT, index=False)

print("\n💾 Saved:")
print(SLIDING_OUT)

In [ ]:
# 15 — FULL GENERATIVE RECONSTRUCTION (NO TRUE STEP, KAGGLE-DEPLOYABLE)

import numpy as np
import pandas as pd
import gc

group_cols = ["target_id", "chain", "copy"]
feature_cols = ["dx_norm", "dy_norm", "dz_norm", "curvature", "turn_angle", "step"]

WINDOW_SIZE = 4
STRIDE = 1

recon_rows = []
total_groups = df_eval.groupby(group_cols).ngroups
processed = 0

print(f"🚀 Evaluating {total_groups} chains in fully generative mode")
print(f"Window size = {WINDOW_SIZE}, stride = {STRIDE}")

for (target_id, chain, copy), group in df_eval.groupby(group_cols):
    processed += 1

    if processed % 25 == 0:
        print(f"⏳ Evaluated {processed}/{total_groups}")

    group = group.sort_values("residue_index").reset_index(drop=True)

    if len(group) < WINDOW_SIZE:
        continue

    coords_true = group[["x", "y", "z"]].values.astype(np.float32)
    Xg = group[feature_cols].values.astype(np.float32)

    pred_dx_g = model_dx.predict(Xg).astype(np.float32)
    pred_dy_g = model_dy.predict(Xg).astype(np.float32)
    pred_dz_g = model_dz.predict(Xg).astype(np.float32)

    n = len(group)

    coord_sum = np.zeros((n, 3), dtype=np.float64)
    coord_count = np.zeros(n, dtype=np.float64)

    window_starts = list(range(0, n - WINDOW_SIZE + 1, STRIDE))
    if window_starts[-1] != n - WINDOW_SIZE:
        window_starts.append(n - WINDOW_SIZE)

    for start in window_starts:
        end = start + WINDOW_SIZE

        # Anchor each window at its true start only for evaluation alignment.
        # For test-time deployment, this anchor must come from your chosen initialization strategy.
        window_coords = [coords_true[start].astype(np.float32).copy()]

        for i in range(start + 1, end):
            step_vec = np.array(
                [pred_dx_g[i - 1], pred_dy_g[i - 1], pred_dz_g[i - 1]],
                dtype=np.float32
            )

            if not np.all(np.isfinite(step_vec)):
                step_vec = np.zeros(3, dtype=np.float32)

            coords_next = window_coords[-1] + step_vec
            window_coords.append(coords_next)

        window_coords = np.asarray(window_coords, dtype=np.float32)

        coord_sum[start:end] += window_coords.astype(np.float64)
        coord_count[start:end] += 1.0

    uncovered = coord_count == 0
    if np.any(uncovered):
        coord_sum[uncovered] = coords_true[uncovered]
        coord_count[uncovered] = 1.0

    coords_pred = (coord_sum / coord_count[:, None]).astype(np.float32)

    point_error = np.linalg.norm(coords_pred - coords_true, axis=1)

    recon_rows.append({
        "target_id": target_id,
        "chain": chain,
        "copy": copy,
        "residue_count": int(len(group)),
        "window_size": int(WINDOW_SIZE),
        "stride": int(STRIDE),
        "mean_error": float(point_error.mean()),
        "median_error": float(np.median(point_error)),
        "p90_error": float(np.percentile(point_error, 90)),
    })

    del pred_dx_g, pred_dy_g, pred_dz_g, coords_pred, point_error, Xg, coords_true, coord_sum, coord_count
    gc.collect()

df_recon_generative = pd.DataFrame(recon_rows)

print("\n🧬 FULL GENERATIVE RECONSTRUCTION RESULTS")
print("Chains evaluated:", len(df_recon_generative))
print("Mean error:", float(df_recon_generative["mean_error"].mean()))
print("Median error:", float(df_recon_generative["median_error"].median()))
print("P90:", float(np.percentile(df_recon_generative["mean_error"], 90)))

display(df_recon_generative.head(10))

GENERATIVE_OUT = "/kaggle/working/HELIX_GENERATIVE_RECON_V1.csv"
df_recon_generative.to_csv(GENERATIVE_OUT, index=False)

print("\n💾 Saved:")
print(GENERATIVE_OUT)

In [ ]:
# 16 — FULL KAGGLE SUBMISSION PIPELINE (SEQUENCE → STRUCTURE)

import numpy as np
import pandas as pd
import gc

# =========================
# LOAD TEST DATA
# =========================
df_test = pd.read_csv(TEST_SEQS_PATH)

if "target_id" not in df_test.columns:
    df_test["target_id"] = df_test["ID"].astype(str)

print("✅ Test rows:", len(df_test))

# =========================
# PARAMETERS
# =========================
WINDOW_SIZE = 4
STRIDE = 1

# =========================
# HELPER: BUILD FEATURES FROM PREDICTED TRAJECTORY
# =========================
def build_features_from_coords(coords):
    coords = np.asarray(coords, dtype=np.float32)
    n = len(coords)

    dx = np.zeros(n, dtype=np.float32)
    dy = np.zeros(n, dtype=np.float32)
    dz = np.zeros(n, dtype=np.float32)
    step = np.zeros(n, dtype=np.float32)

    dx[1:] = coords[1:,0] - coords[:-1,0]
    dy[1:] = coords[1:,1] - coords[:-1,1]
    dz[1:] = coords[1:,2] - coords[:-1,2]

    step[1:] = np.sqrt(dx[1:]**2 + dy[1:]**2 + dz[1:]**2)

    # normalize direction
    dx_norm = np.zeros(n, dtype=np.float32)
    dy_norm = np.zeros(n, dtype=np.float32)
    dz_norm = np.zeros(n, dtype=np.float32)

    valid = step > 1e-6
    dx_norm[valid] = dx[valid] / step[valid]
    dy_norm[valid] = dy[valid] / step[valid]
    dz_norm[valid] = dz[valid] / step[valid]

    # curvature (cosine of angle)
    curvature = np.zeros(n, dtype=np.float32)
    turn_angle = np.zeros(n, dtype=np.float32)

    for i in range(2, n):
        v1 = np.array([dx[i-1], dy[i-1], dz[i-1]])
        v2 = np.array([dx[i], dy[i], dz[i]])

        n1 = np.linalg.norm(v1)
        n2 = np.linalg.norm(v2)

        if n1 > 1e-6 and n2 > 1e-6:
            cos_angle = np.dot(v1, v2) / (n1 * n2)
            cos_angle = np.clip(cos_angle, -1.0, 1.0)
            curvature[i] = cos_angle
            turn_angle[i] = np.arccos(cos_angle)

    df_feat = pd.DataFrame({
        "dx_norm": dx_norm,
        "dy_norm": dy_norm,
        "dz_norm": dz_norm,
        "curvature": curvature,
        "turn_angle": turn_angle,
        "step": step
    })

    return df_feat.fillna(0.0)


# =========================
# MAIN GENERATION LOOP
# =========================
submission_rows = []

print("🚀 Generating structures...")

for idx, row in df_test.iterrows():

    target_id = row["target_id"]
    seq = row["sequence"]

    L = len(seq)

    # initial straight-line bootstrap
    coords = np.zeros((L, 3), dtype=np.float32)
    for i in range(1, L):
        coords[i] = coords[i-1] + np.array([5.7, 0, 0], dtype=np.float32)

    # iterative refinement (2 passes)
    for _ in range(2):

        df_feat = build_features_from_coords(coords)
        X = df_feat[["dx_norm","dy_norm","dz_norm","curvature","turn_angle","step"]].values.astype(np.float32)

        pred_dx = model_dx.predict(X).astype(np.float32)
        pred_dy = model_dy.predict(X).astype(np.float32)
        pred_dz = model_dz.predict(X).astype(np.float32)

        # sliding window stitching
        coord_sum = np.zeros_like(coords, dtype=np.float64)
        coord_count = np.zeros(L, dtype=np.float64)

        starts = list(range(0, L - WINDOW_SIZE + 1, STRIDE))
        if starts[-1] != L - WINDOW_SIZE:
            starts.append(L - WINDOW_SIZE)

        for start in starts:
            end = start + WINDOW_SIZE

            window_coords = [coords[start].copy()]

            for i in range(start+1, end):
                step_vec = np.array([
                    pred_dx[i-1],
                    pred_dy[i-1],
                    pred_dz[i-1]
                ], dtype=np.float32)

                if not np.all(np.isfinite(step_vec)):
                    step_vec = np.zeros(3, dtype=np.float32)

                window_coords.append(window_coords[-1] + step_vec)

            window_coords = np.asarray(window_coords)

            coord_sum[start:end] += window_coords
            coord_count[start:end] += 1

        coords = (coord_sum / coord_count[:, None]).astype(np.float32)

    # =========================
    # STORE OUTPUT
    # =========================
    for i in range(L):
        submission_rows.append({
            "ID": f"{target_id}_{i+1}",
            "x": coords[i,0],
            "y": coords[i,1],
            "z": coords[i,2]
        })

    if idx % 10 == 0:
        print(f"⏳ Processed {idx}/{len(df_test)}")

    gc.collect()

# =========================
# SAVE SUBMISSION
# =========================
df_submission = pd.DataFrame(submission_rows)

SUB_PATH = "/kaggle/working/submission.csv"
df_submission.to_csv(SUB_PATH, index=False)

print("\n✅ SUBMISSION FILE CREATED")
print(SUB_PATH)
print("Rows:", len(df_submission))

display(df_submission.head())

In [ ]:
sample = pd.read_csv(SAMPLE_SUB_PATH)
print("Sample rows:", len(sample))
print("Your rows:", len(df_submission))

In [ ]:
print(df_submission.isna().sum())
print(np.isfinite(df_submission[["x","y","z"]]).all())
